# Exploratory supplement: units kept, convergence re-runs, H4 off-resonance covariate, dense checkerboard

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

import figs   # figs.py, next to this notebook

# Pipeline outputs (../pipeline/run_pipeline.sh): set RR_DERIV, or edit here
DERIV = Path(os.environ.get("RR_DERIV", "/path/to/derivatives"))
A = DERIV / "analysis"
METRICS = ("FA", "MD", "MK")
SCANNERS = ["Ingenia32", "Premier48", "Premier21", "Prisma64", "Prisma32"]
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 60)

## Units kept (Table S7)

Units excluded as unphysical, separately for each metric: FA ≤ 0, MD ≤ 0, MK ≤ 0 or MK > 3, in either arm, in any relevant session. Mean (range) per subject (H1), per subject–scanner pair (H2, H4) or common to all subjects (H3); % of all candidate units. Voxel-wise: one white-matter voxel in eight. ROI-wise: ROIs with at least one valid voxel.

In [2]:
counts = json.loads((A / "inputs" / "counts.json").read_text())
units, tables = counts["units"], counts["tables"]
N_ROI = 1298
rows = []
for h in ("H1", "H2", "H3", "H4"):
    for lv in "ab":
        row = {"Model": h + lv}
        for m in METRICS:
            if h == "H3":
                u = units[f"H3{lv}/common/{m}"]
                cand, kept = [u["candidates"]], [u["kept_all_subjects"]]
            elif lv == "a":
                u = [v for k, v in units.items()
                     if k.split("/")[0] == h and k.split("/")[-1] == m and "checkerboard_voxels" in v]
                cand, kept = [v["checkerboard_voxels"] for v in u], [v["checkerboard_kept"] for v in u]
            else:
                if h == "H1":
                    t = pd.read_csv(A / "inputs" / f"h1b_{m}.csv.gz", usecols=["subject", "den"])
                    kept = t[t.den == 0].groupby("subject").size().tolist()
                else:
                    kept = [v["rows"] for k, v in tables.items() if k.startswith(f"{h.lower()}b_") and k.endswith(f"_{m}")]
                cand = [N_ROI] * len(kept)
            row["Candidate units"] = f"{np.mean(cand):,.0f}" + (f" ({min(cand):,}–{max(cand):,})" if min(cand) < max(cand) else "")
            row[f"{m} kept"] = f"{np.mean(kept):,.0f} ({100 * sum(kept) / sum(cand):.1f}%)"
        rows.append(row)
pd.DataFrame(rows).set_index("Model")

,Candidate units,FA kept,MD kept,MK kept
Model,,,,
H1a,"8,125 (6,301–9,397)","7,781 (95.8%)","7,841 (96.5%)","7,230 (89.0%)"
H1b,"1,298","1,257 (96.9%)","1,259 (97.0%)","1,253 (96.6%)"
H2a,"8,376 (7,615–9,397)","7,978 (95.2%)","8,046 (96.1%)","7,303 (87.2%)"
H2b,"1,298","1,264 (97.4%)","1,266 (97.5%)","1,261 (97.1%)"
H3a,"10,050","9,163 (91.2%)","9,031 (89.9%)","2,668 (26.5%)"
H3b,"1,298","1,157 (89.1%)","1,157 (89.1%)","1,143 (88.1%)"
H4a,"8,369 (7,624–9,271)","8,294 (99.1%)","8,294 (99.1%)","7,966 (95.2%)"
H4b,"1,298","1,275 (98.2%)","1,275 (98.2%)","1,273 (98.1%)"


## Fits that did not meet the convergence criteria: re-run with four times as many draws (Tables S15a, S15b)

In [3]:
out = []
for h in ("H2", "H3"):
    t = figs.results_table(DERIV, h, rerun=True)
    key = [c for c in ("Hypothesis", "pair") if c in t]
    out.append(t[t.groupby(key).run.transform(lambda r: (r != "").any())])
for t in out:
    print(t.drop(columns=[c for c in t if t[c].isna().all()]).to_string(index=False), "\n")

       Hypothesis      pair      run        alpha0 [95% CrI] BF10 converged                verdict
H2b – FA ROI-wise sub-16745          -0.409 [-0.459, -0.355] >100        no denoising changes bias
H2b – FA ROI-wise sub-16745 x4 draws -0.412 [-0.462, -0.356] >100        no denoising changes bias
H2b – FA ROI-wise sub-16766          +0.636 [+0.582, +0.689] >100        no denoising changes bias
H2b – FA ROI-wise sub-16766 x4 draws +0.634 [+0.578, +0.689] >100       yes denoising changes bias
H2b – FA ROI-wise sub-16975          -0.625 [-0.679, -0.571] >100        no denoising changes bias
H2b – FA ROI-wise sub-16975 x4 draws -0.624 [-0.678, -0.571] >100        no denoising changes bias
H2b – MD ROI-wise sub-15320          -0.396 [-0.453, -0.339] >100        no denoising changes bias
H2b – MD ROI-wise sub-15320 x4 draws -0.392 [-0.449, -0.339] >100        no denoising changes bias
H2b – MD ROI-wise sub-16975          -0.363 [-0.417, -0.312] >100        no denoising changes bias
H2b – MD R

## H4 with z-scored |off-resonance| (Table S17)

Registered fits: |off-resonance| in Hz. Exploratory: |off-resonance| z-scored within pair. β1 is shown per SD of the covariate.

In [4]:
VAR = {"|off| Hz (main)": ("results", "off_hz"), "|off| z-scored": ("results_h4_abs_z", "off_z")}
info = figs.pair_info(DERIV)


def verdict(b1, bf):
    if bf < 0.1:
        return "null"
    return "inconclusive" if bf <= 10 else "supports H4" if b1 > 0 else "opposite"


rows = []
for lv, level in (("a", "voxel"), ("b", "ROI")):
    for m in METRICS:
        for q in info:
            tab = pd.read_csv(A / "inputs" / f"h4{lv}_{q}_{m}.csv.gz", usecols=["off_hz", "off_z"])
            for name, (d, col) in VAR.items():
                r = json.loads((A / d / f"H4{lv}_{q}_{m}" / "result.json").read_text())
                rows.append(dict(covariate=name, Hypothesis=figs.hyp_label("H4", level, m), level=level, metric=m, pair=q,
                                 scanner=info[q]["repeat_scanner"].split("_")[0], beta1_sd=r["mean_beta1"] * tab[col].std(ddof=0),
                                 BF10=r["BF10"], verdict=verdict(r["mean_beta1"], r["BF10"]), converged=r["converged"]))
h4 = pd.DataFrame(rows)
print("all fits converged:", bool(h4.converged.all()))
n = h4.groupby(["covariate", "verdict"], sort=False).size().unstack(fill_value=0)[["supports H4", "null", "opposite", "inconclusive"]]
main = h4[h4.covariate == "|off| Hz (main)"].set_index(["level", "metric", "pair"]).verdict
n["same verdict as main"] = [int((g.set_index(["level", "metric", "pair"]).verdict == main).sum())
                             for _, g in h4.groupby("covariate", sort=False)]
s4 = main[main == "supports H4"].index
n["of the tests supporting H4 (main), still supporting"] = [
    int((g.set_index(["level", "metric", "pair"]).verdict.loc[s4] == "supports H4").sum()) for _, g in h4.groupby("covariate", sort=False)]
n

all fits converged: True


verdict,supports H4,null,opposite,inconclusive,same verdict as main,"of the tests supporting H4 (main), still supporting"
covariate,,,,,,
|off| Hz (main),5,11,12,2,30,5
|off| z-scored,5,10,12,3,29,5


In [5]:
h4["cell"] = [f"{b:+.3f}, BF {'>100' if bf > 100 else f'{bf:.2g}'} ({v})" for b, bf, v in zip(h4.beta1_sd, h4.BF10, h4.verdict)]
for name in VAR:
    print(name)
    print(h4[h4.covariate == name].pivot(index="Hypothesis", columns="scanner", values="cell")[SCANNERS].to_string(), "\n")

|off| Hz (main)
scanner                                   Ingenia32                       Premier48                   Premier21                    Prisma64                    Prisma32
Hypothesis                                                                                                                                                             
H4a – FA voxel-wise     +0.027, BF 27 (supports H4)       +0.005, BF 0.00052 (null)     -0.012, BF 0.002 (null)   -0.003, BF 0.00046 (null)   -0.006, BF 0.00061 (null)
H4a – MD voxel-wise         +0.010, BF 0.048 (null)      -0.041, BF >100 (opposite)  -0.037, BF >100 (opposite)  -0.018, BF >100 (opposite)  -0.017, BF >100 (opposite)
H4a – MK voxel-wise   +0.047, BF >100 (supports H4)      -0.074, BF >100 (opposite)  -0.158, BF >100 (opposite)  -0.088, BF >100 (opposite)  -0.121, BF >100 (opposite)
H4b – FA ROI-wise     +0.063, BF >100 (supports H4)     +0.040, BF 12 (supports H4)   +0.004, BF 0.00056 (null)   -0.005, BF 0.00058 (null)    -

In [6]:
w = h4.copy()
w["v"] = w.verdict.replace({"supports H4": "H4"})
w = w.pivot(index=["Hypothesis", "scanner"], columns="covariate", values="v")
w["cell"] = w["|off| Hz (main)"] + " / " + w["|off| z-scored"]
w.cell.unstack("scanner")[SCANNERS]

scanner,Ingenia32,Premier48,Premier21,Prisma64,Prisma32
Hypothesis,,,,,
H4a – FA voxel-wise,H4 / H4,null / null,null / null,null / null,null / null
H4a – MD voxel-wise,null / inconclusive,opposite / opposite,opposite / opposite,opposite / opposite,opposite / opposite
H4a – MK voxel-wise,H4 / H4,opposite / opposite,opposite / opposite,opposite / opposite,opposite / opposite
H4b – FA ROI-wise,H4 / H4,H4 / H4,null / null,null / null,null / null
H4b – MD ROI-wise,H4 / H4,opposite / opposite,null / null,null / null,null / null
H4b – MK ROI-wise,inconclusive / inconclusive,inconclusive / inconclusive,opposite / opposite,opposite / opposite,opposite / opposite


## Dense checkerboard (Table S18)

The voxel-wise models with one white-matter voxel in eight (main analysis) and with the dense checkerboard (about half of the voxels). β1: ×10⁻³ per Hz. BF10; for H3a, P(ΔICC > 0). † did not meet the convergence criteria.

In [7]:
RES = {"1 in 8": A / "results", "dense": A / "results_dense"}
RULE = {"H1": lambda b: b < 0, "H2": lambda b: True, "H4": lambda b: b > 0}
YES = {"H1": "supports H1", "H2": "denoising changes bias", "H4": "H4"}


def bf_text(x):
    return ">100" if x > 100 else f"{x:.1f}" if x >= 1 else f"{x:.3f}" if x >= 0.001 else "<0.001"


def fit_cells(run, d, job, h, target):
    r = json.loads((d / job / "result.json").read_text())
    s = pd.read_csv(d / job / "summary.csv").set_index("param").loc[target]
    k, q = (1e3, 2) if h == "H4" else (1, 3)
    est = f"{k * s['mean']:+.{q}f} [{k * s.q2_5:+.{q}f}, {k * s.q97_5:+.{q}f}]" + ("" if r["converged"] else " †")
    b, bf = r[f"mean_{target}"], r["BF10"]
    v = "null" if bf < 0.1 else "inconclusive" if bf <= 10 else YES[h] if RULE[h](b) else "opposite"
    return {f"{run}: estimate [95% CrI]": est, f"{run}: BF10 or P": bf_text(bf), f"{run}: verdict": v}


def icc_cells(run, d, m):
    r = json.loads((d / f"H3a_{m}_delta_icc.json").read_text())
    p = r["p_delta_gt0"]
    est = f"{r['mean_delta']:+.3f} [{r['q2_5']:+.3f}, {r['q97_5']:+.3f}]" + ("" if r["converged"] else " †")
    return {f"{run}: estimate [95% CrI]": est, f"{run}: BF10 or P": f"{p:.2f}",
            f"{run}: verdict": "supports H3" if p > 0.95 else "null" if p < 0.05 else "inconclusive"}


order = sorted(info, key=lambda q: SCANNERS.index(info[q]["repeat_scanner"].split("_")[0]))
rows = []
for h, target in (("H1", "alpha1"), ("H2", "alpha0"), ("H3", None), ("H4", "beta1")):
    for m in METRICS:
        for q in (order if h in ("H2", "H4") else [None]):
            row = {"Test": f"{h}a {m}", "Pair": info[q]["repeat_scanner"].split("_")[0] if q else "–"}
            for run, d in RES.items():
                row.update(icc_cells(run, d, m) if h == "H3" else
                           fit_cells(run, d, f"{h}a_{q}_{m}" if q else f"{h}a_{m}", h, target))
            rows.append(row)
s18 = pd.DataFrame(rows)
same = s18["1 in 8: verdict"] == s18["dense: verdict"]
print(f"same verdict: {same.sum()} of {len(s18)}; different: "
      + "; ".join(f"{t} {p}" for t, p in s18.loc[~same, ["Test", "Pair"]].values))
with pd.option_context("display.max_rows", None):
    display(s18)

same verdict: 35 of 36; different: H4a MD Ingenia32


,Test,Pair,1 in 8: estimate [95% CrI],1 in 8: BF10 or P,1 in 8: verdict,dense: estimate [95% CrI],dense: BF10 or P,dense: verdict
0,H1a FA,–,"-0.036 [-0.046, -0.026]",>100,supports H1,"-0.038 [-0.043, -0.033]",>100,supports H1
1,H1a MD,–,"-0.074 [-0.084, -0.064]",>100,supports H1,"-0.075 [-0.079, -0.070]",>100,supports H1
2,H1a MK,–,"-0.425 [-0.436, -0.415]",>100,supports H1,"-0.424 [-0.428, -0.419]",>100,supports H1
3,H2a FA,Ingenia32,"-0.513 [-0.536, -0.491]",>100,denoising changes bias,"-0.515 [-0.526, -0.503]",>100,denoising changes bias
4,H2a FA,Premier48,"-0.161 [-0.184, -0.138]",>100,denoising changes bias,"-0.161 [-0.172, -0.150]",>100,denoising changes bias
5,H2a FA,Premier21,"-0.333 [-0.355, -0.311]",>100,denoising changes bias,"-0.315 [-0.326, -0.304]",>100,denoising changes bias
6,H2a FA,Prisma64,"+0.287 [+0.264, +0.310]",>100,denoising changes bias,"+0.292 [+0.281, +0.304]",>100,denoising changes bias
7,H2a FA,Prisma32,"+0.429 [+0.408, +0.450]",>100,denoising changes bias,"+0.450 [+0.439, +0.460]",>100,denoising changes bias
8,H2a MD,Ingenia32,"-0.199 [-0.221, -0.176]",>100,denoising changes bias,"-0.196 [-0.207, -0.185]",>100,denoising changes bias
9,H2a MD,Premier48,"-0.147 [-0.169, -0.124]",>100,denoising changes bias,"-0.150 [-0.162, -0.139]",>100,denoising changes bias
